# Toxicity latent probe — cheap, portable submission test

This notebook is the safer starting point for the next competition run. It keeps the successful submission interface from **Linear_probe working**, but makes the representation contract explicit:

- Gemma version: `google/gemma-2-2b`, matching the third-place `Linear_probe working.ipynb` contract (change only if the competition embeddings were produced by another checkpoint).
- Pooling: final non-padding token, with right padding.
- Probe input: one selected hidden-state layer, shape `[n_examples, hidden_size]`.
- Submission ZIP root: exactly `classifier.py` and `trained_probe.joblib`.

The default mode uses deterministic synthetic embeddings and does **not** download Gemma. Its purpose is to test the training, artifact, ZIP, and platform-import path cheaply. Switch `DATA_MODE` to `embeddings` when the official labelled embedding export is available, or to `gemma_text` when you intentionally want to extract Gemma representations from text.

In [ ]:
%pip install -q "numpy>=1.24" "pandas>=2.0" "scikit-learn>=1.4,<2" "scipy>=1.10" joblib matplotlib seaborn
# Only needed for DATA_MODE = 'gemma_text':
%pip install -q "torch>=2.2" "transformers>=4.55" "huggingface_hub>=0.27" "accelerate>=1.2" "safetensors>=0.4" "datasets>=2.20"

## 0 · Configuration

Run the default configuration first. It is local and inexpensive. For a real run, set `DATA_MODE = 'embeddings'`, point `EMBEDDINGS_PATH` and `METADATA_PATH` at the official labelled export, and confirm that its Gemma contract matches `MODEL_ID`, `MODEL_REVISION`, `POOLING`, `MAX_LENGTH`, and `HIDDEN_STATE_INDEX` before submitting.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import platform
import random
import tempfile
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 17
DATA_MODE = "synthetic"  # synthetic | embeddings | gemma_text

# Exact frozen representation contract. Record the revision for a competition run.
# The proven notebook uses Gemma 2 2B. Gemma 3 270M is NOT representation-compatible.
MODEL_ID = "google/gemma-2-2b"
MODEL_REVISION = None  # set to an immutable HF commit hash for the final run
POOLING = "last_non_padding_token"
MAX_LENGTH = 260  # matches the successful Linear_probe working extraction
BATCH_SIZE = 2
TOKEN_ENV_PATH = Path("BMToken.env")
HIDDEN_STATE_INDEX = None  # None = select by cross-validation; integer = force a layer

# Real embedding input. Supported: .npy [n,d] or [n,layers,d], and .npz with
# an 'X'/'embeddings'/'representations' array plus labels in metadata.
EMBEDDINGS_PATH = Path("official_embeddings.npy")
METADATA_PATH = Path("official_embedding_metadata.csv")
GROUP_COLUMN = None  # e.g. "conversation_id" or "author_id" if supplied
TOXICITY_THRESHOLD = 0.5

TEXT_DATA_PATH = None  # CSV with text + label/toxicity, used only in gemma_text mode
MAX_TEXT_ROWS = 200
REPEATS = 2  # increase to 5–10 for the final labelled embedding run
SUBMISSION_VERSION = "v003_2304_numpy"
SUBMISSION_NAME = (f"cheap_test_{SUBMISSION_VERSION}.zip" if DATA_MODE == "synthetic"
                   else f"submission_{SUBMISSION_VERSION}.zip")

random.seed(SEED)
np.random.seed(SEED)
print({"python": platform.python_version(), "numpy": np.__version__,
       "pandas": pd.__version__, "sklearn": sklearn.__version__})

## 1 · Data loaders

The embedding loader accepts either a two-dimensional matrix or all Gemma hidden states. Labels are read from `label`, `target`, `toxic`, or `toxicity`; groups are optional and are passed to grouped validation when present.

In [ ]:
def _first_key(mapping, names):
    for name in names:
        if name in mapping:
            return name
    return None

def _labels_from_metadata(metadata):
    key = _first_key(metadata, ["label", "target", "toxic"])
    if key is not None:
        values = pd.to_numeric(metadata[key], errors="coerce")
        return values.to_numpy(dtype=float)
    if "toxicity" in metadata:
        values = pd.to_numeric(metadata["toxicity"], errors="coerce")
        return (values >= TOXICITY_THRESHOLD).astype(float).to_numpy()
    raise ValueError("Metadata needs label/target/toxic, or toxicity")

def load_embedding_files(embeddings_path, metadata_path):
    embeddings_path = Path(embeddings_path)
    if not embeddings_path.exists():
        raise FileNotFoundError(f"Missing embedding file: {embeddings_path.resolve()}")
    embedded_labels = None
    if embeddings_path.suffix.lower() == ".npz":
        with np.load(embeddings_path, allow_pickle=False) as data:
            key = _first_key(data, ["X", "embeddings", "representations"] )
            if key is None:
                raise ValueError("NPZ needs an X, embeddings, or representations array")
            X = data[key]
            label_key = _first_key(data, ["y", "labels", "label", "target"])
            if label_key is not None:
                embedded_labels = np.asarray(data[label_key])
    else:
        X = np.load(embeddings_path, allow_pickle=False)
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 2:
        X = X[:, None, :]
    if X.ndim != 3:
        raise ValueError(f"Expected [n,d] or [n,layers,d], got {X.shape}")
    if not np.isfinite(X).all():
        raise ValueError("Embedding matrix contains NaN or infinity")
    if metadata_path is not None and Path(metadata_path).exists():
        metadata = pd.read_csv(metadata_path).reset_index(drop=True)
        y = _labels_from_metadata(metadata)
    elif embedded_labels is not None:
        metadata = pd.DataFrame({"label": embedded_labels})
        y = np.asarray(embedded_labels, dtype=float)
    else:
        raise FileNotFoundError("Provide METADATA_PATH or labels inside the NPZ")
    if len(metadata) != len(X) or len(y) != len(X):
        raise ValueError(f"Rows disagree: X={len(X)}, metadata={len(metadata)}, labels={len(y)}")
    keep = np.isfinite(y)
    y = y[keep].astype(int)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError(f"Labels must contain both 0 and 1; got {np.unique(y)}")
    groups = None
    if GROUP_COLUMN is not None and GROUP_COLUMN in metadata:
        groups = metadata.loc[keep, GROUP_COLUMN].fillna("__missing__").astype(str).to_numpy()
    return X[keep], y, metadata.loc[keep].reset_index(drop=True), groups

def make_synthetic_data(seed=SEED, n=160, layers=6, hidden_size=2304):
    """Cheap deterministic smoke data with the platform's 2304-wide contract."""
    rng = np.random.default_rng(seed)
    y = np.tile([0, 1], n // 2)
    X = rng.normal(0, 1, size=(n, layers, hidden_size)).astype(np.float32)
    strengths = np.array([0.2, 0.5, 1.0, 1.8, 1.2, 0.7])[:layers]
    X[:, :, :4] += ((2 * y - 1)[:, None, None] * strengths[None, :, None]).astype(np.float32)
    metadata = pd.DataFrame({"label": y, "source": "synthetic_smoke"})
    return X, y.astype(int), metadata, None

## 2 · Optional Gemma 3 representation extraction

Gemma is gated. Without signing in, this notebook can still run in synthetic or embedding mode, and it can use a locally cached checkpoint. To download the configured Gemma checkpoint, accept Google's Gemma terms on Hugging Face and set `HF_TOKEN` (or log in with `huggingface-cli login`) in the runtime. The token is read from the environment and never written into the notebook.

In [ ]:
def load_gemma():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    load_hf_token()
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = (torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported()
             else torch.float32)
    kwargs = {"revision": MODEL_REVISION, "token": hf_token, "low_cpu_mem_usage": True}
    kwargs = {k: v for k, v in kwargs.items() if v is not None}
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kwargs)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, **kwargs)
    except OSError as exc:
        raise RuntimeError(
            f"Could not load {MODEL_ID}. It is gated: accept the model terms and set HF_TOKEN,"
            " or use a previously cached checkpoint."
        ) from exc
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(device).eval().requires_grad_(False)
    print(f"Loaded {MODEL_ID} | device={device} | dtype={dtype}")
    return tokenizer, model, device

def load_hf_token(path=TOKEN_ENV_PATH):
    """Load HF_TOKEN locally without printing it or adding it to source."""
    if os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN"):
        return True
    path = Path(path)
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == "HF_TOKEN":
            value = value.strip().strip("\"'")
            if value:
                os.environ["HF_TOKEN"] = value
                return True
    return False

def extract_last_token_representations(texts, tokenizer, model, device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    import torch
    batches = []
    for start in range(0, len(texts), batch_size):
        batch = list(texts[start:start + batch_size])
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        with torch.inference_mode():
            output = model(**encoded, output_hidden_states=True, use_cache=False, return_dict=True)
        last_position = encoded["attention_mask"].sum(dim=1) - 1
        batch_index = torch.arange(len(batch), device=device)
        pooled = torch.stack([state[batch_index, last_position] for state in output.hidden_states], dim=1)
        batches.append(pooled.float().cpu().numpy())
    return np.concatenate(batches, axis=0)

In [ ]:
if DATA_MODE == "synthetic":
    representations, y, metadata, groups = make_synthetic_data()
elif DATA_MODE == "embeddings":
    representations, y, metadata, groups = load_embedding_files(EMBEDDINGS_PATH, METADATA_PATH)
elif DATA_MODE == "gemma_text":
    if TEXT_DATA_PATH is not None and Path(TEXT_DATA_PATH).exists():
        raw = pd.read_csv(TEXT_DATA_PATH).head(MAX_TEXT_ROWS).copy()
        if "label" not in raw and "toxicity" in raw:
            raw["label"] = (raw["toxicity"] >= TOXICITY_THRESHOLD).astype(int)
        if "text" not in raw or "label" not in raw:
            raise ValueError("TEXT_DATA_PATH CSV needs text and label/toxicity columns")
    else:
        from datasets import load_dataset
        raw = load_dataset("google/civil_comments", split=f"train[:{MAX_TEXT_ROWS}]").to_pandas()
        raw["label"] = (raw["toxicity"] >= TOXICITY_THRESHOLD).astype(int)
    raw = raw.dropna(subset=["text", "label"]).reset_index(drop=True)
    if set(raw["label"].astype(int).unique()) != {0, 1}:
        raise ValueError("Text data must contain both binary classes")
    tokenizer, model, device = load_gemma()
    representations = extract_last_token_representations(raw["text"].tolist(), tokenizer, model, device)
    y = raw["label"].astype(int).to_numpy()
    metadata = raw
    groups = raw[GROUP_COLUMN].astype(str).to_numpy() if GROUP_COLUMN in raw else None
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")

assert representations.ndim == 3
assert representations.shape[0] == len(y)
assert np.isfinite(representations).all()
print("Representations:", representations.shape, "labels:", np.bincount(y))
print("Approximate cache size: %.1f MB" % (representations.nbytes / 2**20))

## 3 · Select a layer without touching a hidden test set

The small linear probe is only used for layer selection. The final artifact is fitted later on every labelled row. The selection score averages AUROC, average precision, and balanced accuracy so a lucky threshold does not dominate.

In [ ]:
def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.05, class_weight="balanced", max_iter=4000, random_state=SEED),
    )

def layer_cv(X, y):
    minority = int(np.bincount(y).min())
    if minority < 2:
        raise ValueError("Need at least two examples per class")
    cv = StratifiedKFold(n_splits=min(5, minority), shuffle=True, random_state=SEED)
    rows = []
    for layer in range(X.shape[1]):
        p = np.zeros(len(y), dtype=float)
        for train_idx, valid_idx in cv.split(X[:, layer, :], y):
            model = make_probe().fit(X[train_idx, layer, :], y[train_idx])
            p[valid_idx] = model.predict_proba(X[valid_idx, layer, :])[:, 1]
        pred = (p >= 0.5).astype(int)
        auroc = roc_auc_score(y, p)
        ap = average_precision_score(y, p)
        bal = balanced_accuracy_score(y, pred)
        rows.append({"layer": layer, "auroc": auroc, "average_precision": ap,
                     "balanced_accuracy": bal, "selection_score": np.mean([auroc, ap, bal])})
    return pd.DataFrame(rows).sort_values("selection_score", ascending=False).reset_index(drop=True)

layer_metrics = layer_cv(representations, y)
best_layer = int(HIDDEN_STATE_INDEX if HIDDEN_STATE_INDEX is not None else layer_metrics.iloc[0]["layer"])
if not 0 <= best_layer < representations.shape[1]:
    raise ValueError(f"HIDDEN_STATE_INDEX {best_layer} outside 0..{representations.shape[1]-1}")
display(layer_metrics.round(4))
print(f"Selected layer: {best_layer}")

## 4 · Fit the competition artifact

This uses the hardened `competition_probe.py` ensemble. It does all fitting offline. Group IDs should be supplied whenever related comments can share an author, conversation, template, or source; otherwise ordinary stratified folds are used.

In [ ]:
from competition_probe import fit_competition_ensemble, predict_artifact, make_portable_artifact

X_final = representations[:, best_layer, :].astype(np.float32)
artifact = fit_competition_ensemble(X_final, y, groups=groups, seed=SEED, repeats=REPEATS)
artifact["representation_contract"] = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "hidden_state_index": best_layer,
    "hidden_state_semantics": "0=embedding output; k=output after transformer block k",
    "pooling": POOLING,
    "max_length": MAX_LENGTH,
}
artifact["n_train"] = int(len(y))
artifact["data_mode"] = DATA_MODE
# Convert before saving: sklearn estimator pickles are not portable across
# the local 1.9.0 environment and Codabench's 1.5.1 runtime.
portable_artifact = make_portable_artifact(artifact, X_final, y)
portable_artifact["representation_contract"] = artifact["representation_contract"]
portable_artifact["n_train"] = artifact["n_train"]
portable_artifact["data_mode"] = DATA_MODE
portable_artifact["submission_version"] = SUBMISSION_VERSION
print("Models:", artifact["model_names"])
print("Weights:", np.round(artifact["weights"], 3))
print("OOF score:", round(artifact["oof_score"], 4), "threshold:", round(artifact["threshold"], 3))
print("Training predictions:", np.bincount(predict_artifact(artifact, X_final)))

## 5 · Build and isolate-test the required ZIP

The evaluator receives embeddings, not text, so `classifier.py` deliberately contains no Gemma download and no training call. The model file is loaded relative to the classifier file, which makes the ZIP portable.

In [ ]:
build_dir = Path("submission_build")
build_dir.mkdir(exist_ok=True)
model_path = build_dir / "trained_probe.joblib"
joblib.dump(portable_artifact, model_path, compress=3)
classifier_source = '''import joblib
import numpy as np
from pathlib import Path


class Classifier:
    def __init__(self):
        self.artifact = joblib.load(Path(__file__).with_name("trained_probe.joblib"))

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        expected = int(self.artifact["n_features"])
        if X.ndim != 2 or X.shape[1] != expected:
            raise ValueError(f"Expected [n, {expected}] embeddings; got {X.shape}")
        probability = np.zeros(len(X), dtype=float)
        for weight, model in zip(self.artifact["weights"], self.artifact["models"]):
            z = ((X - model["center"]) / model["scale"])[:, model["support"]]
            z = z @ model["coef"] + float(model["intercept"])
            probability += float(weight) / (1.0 + np.exp(-np.clip(z, -40, 40)))
        return (probability >= float(self.artifact["threshold"])).astype(int)
'''
classifier_path = build_dir / "classifier.py"
classifier_path.write_text(classifier_source, encoding="utf-8")
submission_path = Path(SUBMISSION_NAME)
with zipfile.ZipFile(submission_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(classifier_path, arcname="classifier.py")
    archive.write(model_path, arcname="trained_probe.joblib")

with zipfile.ZipFile(submission_path) as archive:
    names = archive.namelist()
assert names == ["classifier.py", "trained_probe.joblib"], names
print(f"Created {submission_path.resolve()} ({submission_path.stat().st_size / 1024:.1f} KiB)")
print("ZIP root:", names)

In [ ]:
# Isolated import smoke test: exercise the same Classifier interface as the platform.
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with zipfile.ZipFile(submission_path) as archive:
        archive.extractall(tmp)
    spec = importlib.util.spec_from_file_location("submission_classifier", tmp / "classifier.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    classifier = module.Classifier()
    smoke_predictions = classifier.predict(X_final[: min(8, len(X_final))])
    assert smoke_predictions.dtype.kind in "iu"
    assert set(np.unique(smoke_predictions)).issubset({0, 1})
    try:
        classifier.predict(X_final[:, :-1])
    except ValueError:
        pass
    else:
        raise AssertionError("Feature-width validation did not fire")
print("Isolated Classifier smoke test passed:", smoke_predictions.tolist())

## What to change for the real run

1. Obtain the official labelled embedding export and use `DATA_MODE = 'embeddings'`.
2. Verify the embedding width, hidden-state indexing, pooling, and exact Gemma checkpoint used by the competition. The third-place notebook says `google/gemma-2-2b`; the recent Gemma 3 270M notebooks are a different representation contract and should not be mixed with Gemma 2 embeddings.
3. Set `MODEL_REVISION` to the exact Hugging Face commit used for extraction, provide group IDs if available, and increase `REPEATS`.
4. Only then upload `submission.zip`. The default `cheap_test_submission.zip` is a format smoke test, not a competitive model.

The $6 Google Cloud budget is enough for a small Gemma 3 270M extraction/final linear fit, but it is not enough to compensate for missing or mismatched official training embeddings.